MovieLens 100K: Recommender System with Matrix Factorisation (bias terms) + SHAP

In [ ]:
# Step 2: libraries
!pip install pandas numpy scikit-learn shap tqdm matplotlib

In [ ]:
# Step 3: load data from the official URL
import io, os, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")

URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = Path(os.environ.get("ML100K_DIR", "ml-100k"))
if not DATA_DIR.exists():
    print("Downloading MovieLens 100K...")
    r = requests.get(URL, timeout=60); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall()
print("Dataset ready.")

ratings = pd.read_csv(DATA_DIR / "u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
items = pd.read_csv(DATA_DIR / "u.item", sep="|", encoding="latin-1", header=None)
items.columns = ["item_id", "title", "release_date", "video_release_date", "imdb_url"] + \
                [f"genre_{i}" for i in range(19)]

In [ ]:
# Step 4: validation
assert ratings["rating"].between(1, 5).all()
assert not ratings.isna().any().any()
assert not ratings.duplicated(["user_id", "item_id"]).any(), "duplicate (user,item) pairs"
assert set(ratings["item_id"]) <= set(items["item_id"]), "ratings reference unknown items"
assert ratings["user_id"].nunique() <= 1000 and ratings["item_id"].nunique() <= 1700
n_users_all, n_items_all = ratings.user_id.nunique(), ratings.item_id.nunique()
print("Ratings:", len(ratings), "Users:", n_users_all, "Items:", n_items_all)
print("Sparsity: %.4f" % (1 - len(ratings) / (n_users_all * n_items_all)))
print(ratings["rating"].value_counts().sort_index())

In [ ]:
# Step 5: time-aware split 80/10/10
rs = ratings.sort_values("timestamp").reset_index(drop=True)
n = len(rs)
train, val, test = rs.iloc[:int(.8*n)], rs.iloc[int(.8*n):int(.9*n)], rs.iloc[int(.9*n):]
assert train.timestamp.max() <= val.timestamp.min() and val.timestamp.max() <= test.timestamp.min()
print("Split sizes:", len(train), len(val), len(test))
print("Test users unseen in train:", len(set(test.user_id) - set(train.user_id)),
      "| test items unseen in train:", len(set(test.item_id) - set(train.item_id)))

In [ ]:
# Steps 6-8: baselines (global mean, regularised bias model)
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y, float) - p) ** 2)))
def mae(y, p):  return float(np.mean(np.abs(np.asarray(y, float) - p)))

mu = train.rating.mean()
for name, df in [("VAL", val), ("TEST", test)]:
    p = np.full(len(df), mu)
    print(f"Global mean {name}: RMSE={rmse(df.rating, p):.4f} MAE={mae(df.rating, p):.4f}")

lam = 10.0
g = train.groupby("user_id")["rating"]; h = train.groupby("item_id")["rating"]
b_u = ((g.sum() - g.count()*mu) / (g.count()+lam)).to_dict()
b_i = ((h.sum() - h.count()*mu) / (h.count()+lam)).to_dict()
def predict_bias(df):
    return mu + df.user_id.map(b_u).fillna(0).values + df.item_id.map(b_i).fillna(0).values
for name, df in [("VAL", val), ("TEST", test)]:
    p = predict_bias(df)
    print(f"Bias baseline {name}: RMSE={rmse(df.rating, p):.4f} MAE={mae(df.rating, p):.4f}")

In [ ]:
# Step 9: Matrix Factorisation with SGD (+ best-epoch snapshot = early stopping)
from tqdm import trange
user_ids, item_ids = sorted(ratings.user_id.unique()), sorted(ratings.item_id.unique())
u2i = {u: i for i, u in enumerate(user_ids)}; m2i = {m: i for i, m in enumerate(item_ids)}
def to_idx(df): return df.user_id.map(u2i).values, df.item_id.map(m2i).values, df.rating.values
train_u, train_m, train_r = to_idx(train); val_u, val_m, val_r = to_idx(val); test_u, test_m, test_r = to_idx(test)
n_users, n_items = len(user_ids), len(item_ids)

rng = np.random.default_rng(42)
k, lr, reg = 32, 0.01, 0.02
epochs = int(os.environ.get("EPOCHS", 30))
mu_mf = train_r.mean()
bu, bi = np.zeros(n_users), np.zeros(n_items)
P = 0.1 * rng.standard_normal((n_users, k)); Q = 0.1 * rng.standard_normal((n_items, k))
def mf_predict(u, m): return mu_mf + bu[u] + bi[m] + np.sum(P[u] * Q[m], axis=1)

history, best = [], (1e9, None, 0)
for epoch in trange(1, epochs + 1):
    perm = rng.permutation(len(train_r))
    for u, m, r in zip(train_u[perm], train_m[perm], train_r[perm]):
        err = r - (mu_mf + bu[u] + bi[m] + P[u].dot(Q[m]))
        bu[u] += lr * (err - reg * bu[u]); bi[m] += lr * (err - reg * bi[m])
        pu, qi = P[u].copy(), Q[m].copy()
        P[u] += lr * (err * qi - reg * pu); Q[m] += lr * (err * pu - reg * qi)
    tr, va = rmse(train_r, mf_predict(train_u, train_m)), rmse(val_r, mf_predict(val_u, val_m))
    history.append((epoch, tr, va))
    if va < best[0]: best = (va, (bu.copy(), bi.copy(), P.copy(), Q.copy()), epoch)
    if epoch == 1 or epoch % 5 == 0: print(f"Epoch {epoch:02d} train={tr:.4f} val={va:.4f}")

print(f"\nFinal epoch {epochs}: test RMSE={rmse(test_r, mf_predict(test_u, test_m)):.4f}, "
      f"MAE={mae(test_r, mf_predict(test_u, test_m)):.4f}")
overfit = history[-1][2] > best[0] + 1e-3 and history[-1][1] < history[0][1]
print(f"Best val RMSE {best[0]:.4f} at epoch {best[2]} -> overfitting detected: {overfit}")
# Early stopping: restore best-epoch parameters (used for everything below)
bu, bi, P, Q = [x.copy() for x in best[1]]
print(f"Early-stopped MF (epoch {best[2]}): test RMSE={rmse(test_r, mf_predict(test_u, test_m)):.4f}, "
      f"MAE={mae(test_r, mf_predict(test_u, test_m)):.4f}")

In [ ]:
# Step 10: ranking evaluation (relevant = test rating >= 4; seen-in-train items filtered)
train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
test_pos = test[test.rating >= 4].groupby("user_id")["item_id"].apply(set).to_dict()
all_item_idx = np.arange(n_items)

def ndcg_at_k(rel, K):
    dcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rel[:K]))
    idcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(sorted(rel, reverse=True)[:K]))
    return 0.0 if idcg == 0 else float(dcg / idcg)

def recommend_for_user(user_id, K=10):
    uidx = u2i[user_id]
    seen_idx = np.array([m2i[m] for m in train_items_by_user.get(user_id, set())], dtype=int)
    scores = mf_predict(np.full(n_items, uidx), all_item_idx)
    scores[seen_idx] = -1e9
    top = np.argpartition(-scores, K)[:K]; top = top[np.argsort(-scores[top])]
    return [item_ids[i] for i in top], scores[top]

def ranking_eval(K=10):
    precs, recs, ndcgs, cover = [], [], [], set()
    for u in test.user_id.unique():
        gt = test_pos.get(u, set())
        if not gt or u not in u2i: continue
        rec, _ = recommend_for_user(u, K); cover.update(rec)
        hits = [int(m in gt) for m in rec]
        precs.append(np.mean(hits)); recs.append(sum(hits)/len(gt)); ndcgs.append(ndcg_at_k(hits, K))
    return dict(precision=np.mean(precs), recall=np.mean(recs), ndcg=np.mean(ndcgs),
                item_coverage=len(cover)/n_items, users_evaluated=len(precs))
for K in (5, 10, 20): print(f"K={K}:", {a: round(float(b), 4) for a, b in ranking_eval(K).items()})

In [ ]:
# Step 11: diagnostics (cold vs warm users, popularity bias, rating-bucket error)
user_counts = train.groupby("user_id")["rating"].count()
def rmse_subset(df, users):
    d = df[df.user_id.isin(users)]
    if len(d) == 0: return None
    u, m, r = to_idx(d); return rmse(r, mf_predict(u, m)), len(d)
cold = set(user_counts[(user_counts >= 1) & (user_counts <= 10)].index); warm = set(user_counts[user_counts >= 50].index)
print("RMSE test cold users (1-10 train ratings):", rmse_subset(test, cold))
print("RMSE test warm users (>=50 train ratings):", rmse_subset(test, warm))
item_pop = train.groupby("item_id")["rating"].count().to_dict()
pops = [item_pop.get(m, 0) for u in list(test.user_id.unique())[:200] if u in u2i
        for m in recommend_for_user(u, 10)[0]]
print("Avg/median train popularity of recommended items:", np.mean(pops), np.median(pops),
      "| catalogue avg:", np.mean(list(item_pop.values())))
tp = mf_predict(test_u, test_m)
print("Test RMSE by true rating:", {int(r): round(rmse(test_r[test_r == r], tp[test_r == r]), 3) for r in range(1, 6)})

In [ ]:
# Step 12: SHAP with a surrogate. Features = [user_bias, item_bias, P*Q interactions (k)]
from sklearn.linear_model import Ridge
import shap
def mf_feature_matrix(u, m): return np.column_stack([bu[u], bi[m], P[u] * Q[m]])
idx = rng.choice(len(train_r), size=min(20000, len(train_r)), replace=False)
X_surr, y_surr = mf_feature_matrix(train_u[idx], train_m[idx]), mf_predict(train_u[idx], train_m[idx])
surr = Ridge(alpha=1.0).fit(X_surr, y_surr)
print("Surrogate fidelity RMSE (train sample):", rmse(y_surr, surr.predict(X_surr)))
Xt = mf_feature_matrix(test_u, test_m)  # fidelity on held-out test rows too
print("Surrogate fidelity RMSE (test):", rmse(mf_predict(test_u, test_m), surr.predict(Xt)))

explainer = shap.LinearExplainer(surr, shap.maskers.Independent(X_surr))
feature_names = ["user_bias", "item_bias"] + [f"latent_inter_{j}" for j in range(k)]
title_of = dict(zip(items.item_id, items.title))
user_mean = train.groupby("user_id").rating.mean(); item_mean = train.groupby("item_id").rating.mean()

def explain_prediction(user_id, item_id, top_n=6):
    u, m = np.array([u2i[user_id]]), np.array([m2i[item_id]])
    x = mf_feature_matrix(u, m); pred = float(mf_predict(u, m)[0])
    sv = explainer.shap_values(x)[0]; base = float(np.ravel(explainer.expected_value)[0])
    lat = sv[2:].sum()
    print(f"\n=== User {user_id} -> {title_of[item_id]} (item {item_id}) ===")
    print(f"MF pred={pred:.3f} | surrogate={surr.predict(x)[0]:.3f} | SHAP base={base:.3f} | base+sum(SHAP)={base+sv.sum():.3f}")
    print(f"train facts: user mean={user_mean.get(user_id, np.nan):.2f} (n={user_counts.get(user_id, 0)}), "
          f"item mean={item_mean.get(item_id, np.nan):.2f} (n={item_pop.get(item_id, 0)})")
    print(f"user_bias {sv[0]:+.3f} | item_bias {sv[1]:+.3f} | all 32 latent terms combined {lat:+.3f}")
    for nme, v in sorted(zip(feature_names, sv), key=lambda t: -abs(t[1]))[:top_n]: print(f"   {nme:>16}: {v:+.4f}")
    return dict(user=user_id, item=item_id, pred=pred, user_bias=sv[0], item_bias=sv[1], latent=lat)

# Follow-ups: (a) a movie known to be good, (b) a movie predicted low, (c) compare
min_n = 30
good_titles = ["Schindler's List (1993)", "Godfather, The (1972)", "Star Wars (1977)", "Casablanca (1942)", "Shawshank Redemption, The (1994)"]
cand = [(t, i) for i, t in title_of.items() if t in good_titles]
res = {}
if cand:
    t, gi = max(cand, key=lambda c: item_mean.get(c[1], 0))
    rows = test[(test.item_id == gi) & test.user_id.isin(user_counts.index)]
    if len(rows): res["good"] = explain_prediction(int(rows.iloc[0].user_id), gi); print("   actual test rating:", int(rows.iloc[0].rating))
if "good" not in res:  # fallback for any dataset: best-rated well-supported movie
    well_supported = pd.Series(item_pop)[lambda s: s >= min_n].index
    gi = int(item_mean[well_supported].idxmax())
    r0 = test[test.item_id == gi].iloc[0]; res["good"] = explain_prediction(int(r0.user_id), gi)

tdf = test.copy(); tdf["pred"] = tp
tdf = tdf[tdf.item_id.map(item_pop).fillna(0) >= 10]       # items with enough evidence
low = tdf.loc[tdf.pred.idxmin()]
res["low"] = explain_prediction(int(low.user_id), int(low.item_id)); print("   actual test rating:", int(low.rating))

print("\n--- Comparison ---")
print(pd.DataFrame(res).T[["pred", "user_bias", "item_bias", "latent"]].round(3))

# Global view: which SHAP features matter on average (mean |SHAP|, cf. paper Eq. 3)
sv_all = explainer.shap_values(Xt[:2000])
imp = pd.Series(np.abs(sv_all).mean(0), index=feature_names).sort_values(ascending=False)
print("\nGlobal mean|SHAP| top 6:\n", imp.head(6).round(4))
print("Share of total mean|SHAP| from user_bias+item_bias: %.1f%%" % (100 * imp[["user_bias", "item_bias"]].sum() / imp.sum()))